# Breaking the Blend - template for a new galaxy

A clean version of the automated Breaking the Blend workflow (Veenema et al. 2026) **with no galaxy-specific
settings**. Use it for any NIRSpec and/or MIRI/MRS (or other IFS) data: fill in the cells marked **EDIT**,
run, inspect, and refine. Defaults are generic starting points, not tuned values.

| Section | What happens |
|---|---|
| 0 | **EDIT** - name, data folder, redshift, run and figure switches |
| 1 | Find & load cubes, check the AGN position |
| 2 | **EDIT** - emission lines and (optional) per-line settings |
| 3 | **Step 1** - line maps and the dispersion of each component |
| 4 | **EDIT** - **Step 2** - kinematic models: which parameters are fixed / fitted, which lines trace what |
| 5 | **EDIT** - **Step 3** - decomposition: where the model parameters and dispersions come from |
| 6 | **Step 4** - stacked AGN outflow fraction map |
| 7 | **Step 5** - AGN- and star-formation-dominated spectra |

Every slow step is saved in `RESULTS_DIR/GALAXY/` and re-loaded next time (recomputed automatically when its settings
change). Work through the sections in order the first time and look at every figure - the checks are what make the
result trustworthy. See `01_NGC5728.ipynb` for a fully worked example.

## 0. Settings - EDIT

In [ ]:
# =====================================================================
# 0. ANALYSIS SETTINGS
# =====================================================================
GALAXY       = "MyGalaxy"              # analysis name -> results are saved in RESULTS_DIR/GALAXY/
DATA_DIR     = "path/to/cubes"         # folder containing the *_s3d.fits(.gz) cubes (searched recursively)
RESULTS_DIR  = "results"
Z            = None                    # redshift  <-- REQUIRED
DISTANCE_MPC = None                    # optional distance [Mpc] for the physical scale (None = from Z)

# Re-run switches: False -> load saved results if they exist (and their settings are unchanged)
RERUN_LINE_FITS     = False
RERUN_KINEMATICS    = False
RERUN_DECOMPOSITION = False

# Kinematic fitting. "least_squares" is fast for a first pass; switch to "emcee" for final results
# (posteriors / uncertainties), e.g. dict(method="emcee", nwalkers=32, nsteps=20000, burnin=5000, sigma_v="residual")
MCMC = dict(method="least_squares", n_starts=20, sigma_v="residual")

# Figure output: "png", "pdf", "both" or "none"  (saved in RESULTS_DIR/GALAXY/figures/)
SAVE_FIGURES = "both"
FIGURE_DPI   = 200

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
try:
    import breaking_the_blend as btb
except ModuleNotFoundError:   # not installed (pip install -e .): use the copy in this repository
    import sys, os
    sys.path.insert(0, os.path.abspath(".."))
    import breaking_the_blend as btb
from breaking_the_blend import plotting as bplt     # also sets the publication plotting style

if Z is None:
    raise ValueError("Set the redshift Z in Section 0")
proj = btb.Project(GALAXY, data_dir=DATA_DIR, z=Z, distance_mpc=DISTANCE_MPC, results_dir=RESULTS_DIR,
                   save_figures=SAVE_FIGURES, figure_dpi=FIGURE_DPI)

## 1. Data: find the cubes and locate the AGN

All `*s3d.fits*` files under `DATA_DIR` are found and named by band (`ch3-short`, `g395h-f290lp`, ...). Use
`exclude=[...]` to skip files (e.g. background cubes) or `proj.set_cube_files({name: path})` to choose them by hand.

The AGN position of each cube is its (centroided) continuum peak. **Check every cross** in the overview figure; if
one is off the nucleus (e.g. a bright star-forming knot, or a weak AGN), give its pixel position in `AGN_OVERRIDES`.

In [ ]:
files = proj.find_cubes(exclude=["*bkg*"])

In [ ]:
AGN_OVERRIDES = {}            # EDIT if needed: {"cube name": (x, y)} in pixels
PIXEL_SCALE_OVERRIDES = {}    # normally empty: the header pixel size is used

LOAD_ERR = True               # also read the ERR extensions (spectrum errors in Step 5; doubles cube memory)

cubes = proj.load_cubes(agn_overrides=AGN_OVERRIDES, pixel_scale_overrides=PIXEL_SCALE_OVERRIDES, load_err=LOAD_ERR)
fig = bplt.plot_cubes_overview(cubes);  proj.savefig(fig, "cubes_overview")

## 2. Emission lines - EDIT

Start from a broad list; lines not covered by any loaded cube are skipped automatically. After Step 1, **remove lines
that are weak or blended** in your data. Per-line settings (`LINE_SETTINGS`) are only needed where the defaults
fail - look at the spaxel fits in Step 1 first, or test settings quickly with `03_check_line_settings.ipynb`. Useful settings: `line_region` (rest $\mu$m range of the line;
widen it for broad outflows), `fit_window`, `mask_percentile`, `p0_sigma_kms`, `p0_dv_kms`, `p0_amp_frac`, `cube`.
All options: `help(btb.LineSpec)`; the catalogue: `btb.LINE_CATALOGUE`.

In [ ]:
LINES = ["FeII", "ArII", "NeII", "SIII", "ArIII", "SIV", "NeIII", "OIV", "ArVI", "MgIV", "NeV", "NeVI"]

COMMON_SETTINGS = dict(mask_percentile=40)          # applied to every line (40 -> brightest 60% of spaxels)
LINE_SETTINGS = {
    # [Cl II] 14.368 um lies ~1000 km/s from [Ne V] 14.32 um: keep it out of the [Ne V] window
    "NeV": dict(line_region=(14.295, 14.345), fit_window=(14.265, 14.350)),
}
# Example of a full set of per-line settings (those used for NGC 5728 in notebook 01). Uncomment and adapt
# lines as needed - check them first with 03_check_line_settings.ipynb.
# LINE_SETTINGS = {
#     "FeII":  dict(cube="ch1-short",  line_region=(5.320, 5.355),  mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(-2.8, 1.1),  p0_sigma_kms=(112, 225)),
#     "ArII":  dict(cube="ch1-long",   line_region=(6.975, 7.000),  mask_percentile=50,
#                   p0_amp_frac=(0.77, 0.50), p0_dv_kms=(-21.5, 4.3), p0_sigma_kms=(86, 215)),
#     "NeII":  dict(cube="ch3-short",  line_region=(12.800, 12.830), mask_percentile=70,
#                   p0_amp_frac=(0.50, 0.33), p0_dv_kms=(-23.4, 2.3), p0_sigma_kms=(105, 164)),
#     "SIII":  dict(cube="ch4-short",  line_region=(18.650, 18.780), mask_percentile=55,
#                   p0_amp_frac=(0.71, 0.40), p0_dv_kms=(-2.4, 3.2),  p0_sigma_kms=(112, 152)),
#     "ArIII": dict(cube="ch2-medium", line_region=(8.980, 9.015),  mask_percentile=55,
#                   p0_amp_frac=(0.71, 0.43), p0_dv_kms=(-1.7, 1.7),  p0_sigma_kms=(100, 200)),
#     "SIV":   dict(cube="ch2-long",   line_region=(10.485, 10.535), mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 28.5),  p0_sigma_kms=(86, 257)),
#     "NeIII": dict(cube="ch3-long",   line_region=(15.520, 15.600), mask_percentile=60,
#                   p0_amp_frac=(0.77, 0.33), p0_dv_kms=(-3.9, 3.9),  p0_sigma_kms=(106, 173)),
#     "OIV":   dict(cube="ch4-long",   line_region=(25.830, 25.970), mask_percentile=55,
#                   p0_amp_frac=(0.67, 0.33), p0_dv_kms=(0.0, 1.2),   p0_sigma_kms=(81, 127)),
#     "ArVI":  dict(cube="g395h-f290lp", line_region=(4.524, 4.537), mask_percentile=55,
#                   p0_amp_frac=(0.33, 0.50), p0_dv_kms=(-3.3, 3.3),  p0_sigma_kms=(66, 165)),
#     "MgIV":  dict(cube="g395h-f290lp", line_region=(4.481, 4.492), mask_percentile=55,
#                   p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.7, 0.1),  p0_sigma_kms=(134, 267)),
#     "NeV":   dict(cube="ch3-medium", line_region=(14.295, 14.345), fit_window=(14.265, 14.350),  # avoid [Cl II]
#                   mask_percentile=50),
#     "NeVI":  dict(cube="ch2-short",  line_region=(7.645, 7.670),  mask_percentile=50,
#                   p0_amp_frac=(0.33, 0.67), p0_dv_kms=(-0.4, 0.0),  p0_sigma_kms=(78, 176)),
# }
# Note: `cube` names must match your loaded cubes, and settings tuned to one galaxy's line widths
# (line_region, p0_sigma_kms) may not suit another.

lines = btb.get_lines(LINES)
lines = {n: l.update(**{**COMMON_SETTINGS, **LINE_SETTINGS.get(n, {})}) for n, l in lines.items()}
lines = btb.assign_cubes(lines, cubes, drop_missing=True)
proj.save_config(lines=lines)

## 3. Step 1 - measure every line in every spaxel

Continuum subtraction, line flux, masking, single- and double-Gaussian fits and moment maps for every line.
The defaults are conservative: bounded fits (`fitter="trf"`), unweighted (`weighting="none"`; use `"err"` if your cubes
have reliable ERR extensions), started at each spaxel's moment-1 velocity, and the flux percentile computed inside the
field of view.

In [ ]:
FIT_OPTIONS = dict(fitter="trf", weighting="none", p0_center="moment", percentile_over="fov", dilate=2)

line_maps = proj.checkpoint("step1_line_maps", btb.fit_lines, cubes, lines, **FIT_OPTIONS,
                            rerun=RERUN_LINE_FITS, settings=dict(lines=lines, **FIT_OPTIONS))

**Check the fits.** Change `CHECK_LINE` to look at each line in turn. If the double-Gaussian fits look wrong, adjust
that line in `LINE_SETTINGS` (Section 2); it is re-fitted automatically.

In [ ]:
CHECK_LINE = list(line_maps)[len(line_maps) // 2]       # EDIT to inspect other lines
fig = bplt.plot_spaxel_fits(line_maps[CHECK_LINE], n=6);  proj.savefig(fig, f"spaxel_fits_{CHECK_LINE}")

In [ ]:
fig = bplt.plot_line_maps(line_maps, "flux");      proj.savefig(fig, "flux_maps")
fig = bplt.plot_line_maps(line_maps, "velocity");  proj.savefig(fig, "velocity_maps")
fig = bplt.plot_line_maps(line_maps, "sigma");     proj.savefig(fig, "single_gaussian_dispersion_maps")

Use these maps (ordered by IP) to decide, for Section 4, **which lines are dominated by the disc, which by the outflow,
and which show both**. Note the velocity colour range (`vlim=(-v, v)`) can be changed in every map function.

In [ ]:
fig = bplt.plot_line_quantities(line_maps, ("v_n", "v_b"));          proj.savefig(fig, "double_gaussian_velocity_maps")
fig = bplt.plot_line_quantities(line_maps, ("sigma_n", "sigma_b"));  proj.savefig(fig, "double_gaussian_dispersion_maps")
fig = bplt.plot_line_quantities(line_maps, ("mom0", "mom1", "mom2")); proj.savefig(fig, "moment_maps")

### Dispersion of each component

With $\sigma_{\rm outflow} > \sigma_{\rm disc}$ enforced per spaxel, the medians are the default fixed dispersions for
Step 3. For lines with a single component the narrow/broad split is meaningless - check the histograms, and use
`SIGMA_OVERRIDES = {"line": (sigma_disc, sigma_outflow)}` where needed.

In [ ]:
SIGMA_OVERRIDES = {}

sigmas_measured = btb.dispersion_medians(line_maps, sigma_range=(15, 600), overrides=SIGMA_OVERRIDES)
fig = bplt.plot_dispersion_histograms(line_maps, sigmas_measured, xlim=None)
proj.savefig(fig, "dispersion_histograms")
proj.save_config(sigmas_measured=sigmas_measured)

## 4. Step 2 - kinematic models - EDIT

Built-in models (offsets from the AGN in **pc**, angles in degrees, PA **east of north**):

* **thin rotating disc** - `v_disc`, `R_disc`, `inc_disc`, `pa_disc` (PA of the *receding* side);
* **hollow bicone** - `v_out`, `R_out`, `inc_out`, `pa_out` (PA of the *redshifted* cone), `theta_in`, `theta_out`;
* **combined** - $v = \alpha v_{\rm disc} + (1-\alpha) v_{\rm outflow}$, plus `v_sys`.

In `MODEL_PARAMS` a **number fixes** a parameter and a **tuple `(lo, hi)` fits** it with a uniform prior. Any
combination works. **If the inclinations and position angles are known (literature, other tracers), fix them** - a
partial field of view rarely constrains the geometry well. Below everything is free with broad priors as a starting
point. If a fitted model looks mirrored, the PA is 180 deg off.

Keep the `(lo, hi)` ranges **wide** unless you have a reason to restrict them. A fit that ends on the edge of its range
is reported as `<-- at prior limit`: that value is set by the prior rather than the data, so widen that range (or fix
the parameter deliberately). Least-squares errors are approximate (and zero for the opening angles, whose effect is
not smooth); use `method="emcee"` for meaningful uncertainties.

In [ ]:
MODEL_PARAMS = {
    "v_sys":     0.0,           # km/s  (0 if the redshift is right; e.g. (-100, 100) to fit)
    # ---- disc ---------------------------------------------------------------------------------
    "inc_disc":  (10, 80),      # deg   EDIT: fix to a known value if available, e.g. 45.0
    "pa_disc":   (0, 360),      # deg E of N (receding side)
    "v_disc":    (30, 600),     # km/s
    "R_disc":    (20, 2000),    # pc
    # ---- outflow -----------------------------------------------------------------------------
    "inc_out":   (10, 80),      # deg
    "pa_out":    (0, 360),      # deg E of N (redshifted cone)
    "v_out":     (50, 2000),    # km/s
    "R_out":     (50, 3000),    # pc
    "theta_in":  (0, 60),       # deg
    "theta_out": (20, 120),     # deg
    # ---- combined -----------------------------------------------------------------------------
    "alpha":     (0, 1),
}

**Line groups - EDIT after looking at the Step 1 maps.** The initial guess below uses the ionisation potential:
low-IP lines (< 25 eV) for the disc, high-IP lines (> 55 eV) for the outflow, and every line below 90 eV for the
decomposition. Lines with **no** disc component (check the narrow-component maps) should be left out of
`DECOMPOSE_LINES` - they still constrain the outflow.

In [ ]:
DISC_LINES      = [n for n, l in lines.items() if l.ip < 25]     # EDIT
OUTFLOW_LINES   = [n for n, l in lines.items() if l.ip > 55]     # EDIT
DECOMPOSE_LINES = [n for n, l in lines.items() if l.ip < 90]     # EDIT
print("disc:", DISC_LINES, "\noutflow:", OUTFLOW_LINES, "\ndecompose:", DECOMPOSE_LINES)

RUN_KINEMATIC_FITS = True
# For the geometry schematics only: which side of the disc minor axis is nearer to us
# (+1: the side at PA_disc + 90 deg, -1: PA_disc - 90 deg) - set from dust lanes if known.
DISC_NEAR_SIDE = +1
# Geometry fitted in the single-component fits (4.1, 4.2) is averaged and FIXED in the combined fits (4.3),
# which otherwise have too much freedom. Set False to keep it free there too.
FIX_GEOMETRY_IN_COMBINED = True

In [ ]:
GEOM = ("inc_disc", "pa_disc", "inc_out", "pa_out")
free = lambda keys: [k for k in keys if not np.isscalar(MODEL_PARAMS[k])]
disc_fits, outflow_fits, combined_fits = {}, {}, {}

if RUN_KINEMATIC_FITS and DISC_LINES:
    # 4.1 disc-only fits to the disc-dominated lines
    disc_fits = proj.checkpoint("step2a_disc_fits", btb.fit_many, line_maps, DISC_LINES, "disc",
                                params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                settings=dict(lines=DISC_LINES, params=MODEL_PARAMS, **MCMC))
    n = DISC_LINES[0]
    fig = bplt.plot_kinematic_fit(disc_fits[n], line_maps[n]);  proj.savefig(fig, f"disc_fit_{n}")

In [ ]:
if RUN_KINEMATIC_FITS and OUTFLOW_LINES:
    # 4.2 outflow-only fits to the outflow-dominated lines
    outflow_fits = proj.checkpoint("step2b_outflow_fits", btb.fit_many, line_maps, OUTFLOW_LINES, "outflow",
                                   params=MODEL_PARAMS, **MCMC, rerun=RERUN_KINEMATICS,
                                   settings=dict(lines=OUTFLOW_LINES, params=MODEL_PARAMS, **MCMC))
    n = OUTFLOW_LINES[-1]
    fig = bplt.plot_kinematic_fit(outflow_fits[n], line_maps[n]);  proj.savefig(fig, f"outflow_fit_{n}")

In [ ]:
if RUN_KINEMATIC_FITS:
    # 4.3 combined fits to every decomposable line, started near the single-component results
    start = {**btb.mean_params(disc_fits, free(("v_disc", "R_disc") + GEOM[:2])),
             **btb.mean_params(outflow_fits, free(("v_out", "R_out", "theta_in", "theta_out") + GEOM[2:]))}
    COMBINED_PARAMS = dict(MODEL_PARAMS)
    if FIX_GEOMETRY_IN_COMBINED:
        COMBINED_PARAMS.update({k: start[k] for k in GEOM if k in start})
    init = {k: (0.7 * v, 1.3 * v + 1) for k, v in start.items() if k not in GEOM}
    combined_fits = proj.checkpoint("step2c_combined_fits", btb.fit_many, line_maps, DECOMPOSE_LINES,
                                    "combined", params=COMBINED_PARAMS, init=init, **MCMC, rerun=RERUN_KINEMATICS,
                                    settings=dict(lines=DECOMPOSE_LINES, params=COMBINED_PARAMS, init=init, **MCMC))
    btb.fits_table(combined_fits)
    n = DECOMPOSE_LINES[len(DECOMPOSE_LINES) // 2]
    fig = bplt.plot_kinematic_fit(combined_fits[n], line_maps[n]);  proj.savefig(fig, f"combined_fit_{n}")

Parameters against IP (box: 25-75th percentile, whiskers: 16-84th, line: median; least-squares fits show error bars).
Parameters that are roughly constant are averaged for the decomposition; any that change systematically with IP (in
NGC 5728: $R_{\rm out}$, $\alpha$) get a linear trend. **Check the plot** before trusting a trend.

In [ ]:
if RUN_KINEMATIC_FITS:
    fig = bplt.plot_params_vs_ip(combined_fits, ["R_out", "alpha", "v_disc", "v_out"], style="box")
    proj.savefig(fig, "params_vs_IP")
    FIXED_IN_FITS = {k: v for k, v in COMBINED_PARAMS.items() if np.isscalar(v)}          # fixed values
    FIT_MEANS = btb.mean_params(combined_fits, [k for k in COMBINED_PARAMS if k != "alpha"])  # fitted: averaged
    R_OUT_TREND = (btb.trend_with_ip(combined_fits, "R_out")
                   if (not np.isscalar(COMBINED_PARAMS["R_out"]) and len(combined_fits) > 2) else None)

**Best-fit geometry.** Schematic of the disc and bicone from the combined fits (averaged over the lines; cones drawn out
to 2 $R_{\rm out}$). Left: as seen by the observer - parts of the cones behind the disc are drawn as a wire-frame, parts in
front are shaded, yellow lines mark where the cones cross the disc plane. Right: a 3-D view with the line of sight.
Check that it is physically sensible (e.g. the cone that is redshifted points away from us). `DISC_NEAR_SIDE` flips
which side of the disc is nearer to us.

In [ ]:
if RUN_KINEMATIC_FITS:
    FIT_GEOMETRY = btb.mean_geometry(combined_fits)
    fig = btb.plot_geometry(FIT_GEOMETRY, disc_near_side=DISC_NEAR_SIDE, title=f"{GALAXY}: best-fit geometry")
    proj.savefig(fig, "geometry_best_fit")

### Optional: velocity / dispersion maps from other codes (3DBarolo, MOKA3D, ...)

Any component can use a map produced elsewhere instead of the built-in model. Give the AGN pixel position in that map,
its pixel size, and the systemic velocity to subtract if it holds absolute velocities. Keys: `"v_disc"`, `"v_out"`,
`"sigma_disc"`, `"sigma_out"`. Where an imported map is undefined the built-in model / fixed dispersion is used.

In [ ]:
USE_EXTERNAL_MAPS = False
EXTERNAL = {}

if USE_EXTERNAL_MAPS:
    pc_scale = next(iter(cubes.values())).pc_per_arcsec
    EXTERNAL["v_disc"] = btb.load_external_map("path/to/model_velocity.fits",   # EDIT
                                               agn_xy=(0.0, 0.0),                # EDIT: centre in that map
                                               pixel_scale_arcsec=0.2,           # EDIT (or read from the header)
                                               pc_per_arcsec=pc_scale, subtract=0.0)
    lm = line_maps[DECOMPOSE_LINES[0]]
    fig = bplt.plot_velocity_panels({"observed": lm.velocity_map(),
                                     "imported disc": np.where(lm.good, EXTERNAL["v_disc"].on_grid(lm), np.nan)},
                                    lm.extent_pc(), vlim=400)
    proj.savefig(fig, "external_map_check")

## 5. Step 3 - constrained decomposition - EDIT

`DECOMP_PARAMS` selects where each line's model parameters come from:

* `"fit_mean"` (default) - fitted geometry + averages of the combined fits, with $R_{\rm out}$ from its trend with IP
  (set `USE_R_OUT_TREND = False` to use the average instead);
* `"fit_per_line"` - each line's own combined fit (more freedom; check weak lines carefully);
* `"fixed_common"` - your own values in `FIXED_COMMON`, the same for every line;
* `"fixed_per_line"` - your own values per line in `FIXED_LINE_PARAMS` (radii in pc).

`DECOMP_SIGMAS`: `"measured"` (Step 1 medians) or `"fixed"` (`FIXED_SIGMAS`).

In [ ]:
DECOMP_PARAMS   = "fit_mean"     # "fit_mean" | "fit_per_line" | "fixed_common" | "fixed_per_line"
DECOMP_SIGMAS   = "measured"     # "measured" | "fixed"
USE_R_OUT_TREND = True

# Only used with the "fixed_*" options - EDIT with your own values
FIXED_COMMON = dict(v_sys=0.0, inc_disc=None, pa_disc=None, v_disc=None, R_disc=None,    # km/s, deg, pc
                    inc_out=None, pa_out=None, v_out=None, R_out=None, theta_in=None, theta_out=None)
FIXED_LINE_PARAMS = {}           # {"line": {"v_disc": ..., "R_disc": ..., ...}} - overrides FIXED_COMMON per line
FIXED_SIGMAS = {}                # {"line": (sigma_disc, sigma_outflow)} [km/s]

# Optional: decompose spectra re-extracted with other windows / continuum, e.g.
#   {"NeII": dict(window=(12.72, 12.92), line_region=(12.785, 12.835), cont_order=1)}
DECOMP_RECUT = {}

In [ ]:
dmaps = {n: line_maps[n] for n in DECOMPOSE_LINES}
if DECOMP_PARAMS == "fit_mean":
    rel = {"R_out": (lambda ip, a=R_OUT_TREND[0], b=R_OUT_TREND[1]: a * ip + b)} \
          if (USE_R_OUT_TREND and R_OUT_TREND is not None) else None
    line_parameters = btb.line_params(dmaps, {**FIXED_IN_FITS, **FIT_MEANS}, ip_relations=rel)
elif DECOMP_PARAMS == "fit_per_line":
    line_parameters = btb.line_params(dmaps, {}, fits=combined_fits)
elif DECOMP_PARAMS in ("fixed_common", "fixed_per_line"):
    per_line = FIXED_LINE_PARAMS if DECOMP_PARAMS == "fixed_per_line" else None
    line_parameters = btb.line_params(dmaps, FIXED_COMMON, per_line=per_line)
    missing = {n: [k for k, v in p.items() if v is None] for n, p in line_parameters.items()}
    if any(missing.values()):
        raise ValueError(f"Fill in FIXED_COMMON / FIXED_LINE_PARAMS first - missing: {missing}")
else:
    raise ValueError(DECOMP_PARAMS)

if DECOMP_SIGMAS == "fixed":
    sigmas = {n: {"disc": FIXED_SIGMAS[n][0], "outflow": FIXED_SIGMAS[n][1]} for n in DECOMPOSE_LINES}
else:
    sigmas = sigmas_measured

proj.save_config(decomp_params_source=DECOMP_PARAMS, decomp_sigmas_source=DECOMP_SIGMAS,
                 decomposition_params=line_parameters, decomposition_sigmas=sigmas)

In [ ]:
DECOMP_OPTIONS = dict(names=DECOMPOSE_LINES, sigma_mode="fixed", recut=DECOMP_RECUT or None)

decomps = proj.checkpoint("step3_decompositions", btb.decompose_lines, line_maps, line_parameters, sigmas,
                          cubes=cubes, external=EXTERNAL or None, **DECOMP_OPTIONS, rerun=RERUN_DECOMPOSITION,
                          settings=dict(params=line_parameters, sigmas=sigmas, external=sorted(EXTERNAL),
                                        **DECOMP_OPTIONS))

Disc-fraction map of every line, and the residual maps: **coherent residual structure** means the two-component
model does not describe that line (a third component, wrong velocities/dispersions, or a blend).

In [ ]:
fig = bplt.plot_fraction_maps(decomps, "f_disc");  proj.savefig(fig, "disc_fraction_maps")
fig = bplt.plot_residual_maps(decomps);            proj.savefig(fig, "residual_maps")

## 6. Step 4 - stack all lines into one map

Per-line $f_{\rm disc}$ maps on a common parsec grid, weighted by $w = 1 - 2|\langle f_{\rm disc}\rangle - 0.5|$, keeping
points covered by at least `min_lines` lines. `MAP_LIM_PC` / `SCALEBAR_PC` only affect the figure.

In [ ]:
STACK_OPTIONS = dict(scheme="linear", combine="mean", min_lines=2)
MAP_LIM_PC  = None       # e.g. 1500 to zoom the figure
SCALEBAR_PC = None       # e.g. 500 for a scale bar
SF_MAX, AGN_MIN = 0.2, 0.8      # f_out thresholds for disc- / outflow-dominated (also used for the spectra)

stack = btb.stack_fraction_maps(decomps, **STACK_OPTIONS)
print(f"line-to-line scatter in f_disc: {btb.line_to_line_scatter(stack):.2f}")
FRACTIONS = btb.fraction_summary(stack, sf_max=SF_MAX, agn_min=AGN_MIN)
proj.save_config(fraction_summary=FRACTIONS)

fig = bplt.plot_stacked_map(stack, which="f_out", lim_pc=MAP_LIM_PC, scalebar_pc=SCALEBAR_PC, title=GALAXY)
proj.savefig(fig, "outflow_fraction_map")
proj.save(stack, "step4_stacked_map")
proj.export_stacked_map(stack)

**Geometry used in the decomposition** (averaged over the decomposed lines). With per-line parameters this is their
mean; if imported maps replaced a component (Section 4), its drawn geometry is only indicative.

In [ ]:
USED_GEOMETRY = btb.mean_geometry(line_parameters)
fig = btb.plot_geometry(USED_GEOMETRY, disc_near_side=DISC_NEAR_SIDE, title=f"{GALAXY}: geometry used in the decomposition")
proj.savefig(fig, "geometry_used")

**Robustness**: the map should not depend strongly on the weighting or on any single line.

In [ ]:
alternatives = {"linear weights": stack,
                "equal weights": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "scheme": "equal"}, verbose=False),
                "median": btb.stack_fraction_maps(decomps, **{**STACK_OPTIONS, "combine": "median"}, verbose=False)}
for n in DECOMPOSE_LINES:                                       # leave each line out in turn
    alternatives[f"without {n}"] = btb.stack_fraction_maps(decomps, names=[m for m in DECOMPOSE_LINES if m != n],
                                                           **STACK_OPTIONS, verbose=False)
btb.compare_stacks(alternatives, "linear weights")

## 7. Step 5 - AGN and star-forming spectra

Spaxels with $f_{\rm out} > $ `AGN_MIN` give the AGN-dominated spectrum, $f_{\rm out} < $ `SF_MAX` the star-formation-
dominated one (`method="weighted"` weights every spaxel by its fractions instead). Check that both regions contain
spaxels in the extraction plot.

**Errors** (third column of the exported `.txt` files, 1$\sigma$ in Jy, shaded in the plot): `err_method="cube"` adds the
pipeline ERR values of the selected spaxels in quadrature - a lower limit, because neighbouring spaxels of a resampled
cube share noise; `"empirical"` measures the channel-to-channel scatter of each extracted spectrum (includes correlated
noise, inflated next to strong lines); `"max"` takes the larger. The band-to-band scale factors' uncertainty is not included.

In [ ]:
SPECTRA_OPTIONS = dict(method="threshold", agn_min=AGN_MIN, sf_max=SF_MAX, min_spaxels=3,
                       err_method="cube")   # errors: "cube" (ERR, quadrature), "empirical", "max" or None

spectra = btb.extract_all(cubes, stack, **SPECTRA_OPTIONS)
CHECK_BAND = lines[DECOMPOSE_LINES[0]].cube                    # EDIT to look at another band
fig = bplt.plot_extraction_regions(spectra[CHECK_BAND], cubes[CHECK_BAND]);  proj.savefig(fig, "extraction_regions")

In [ ]:
w_agn, f_agn, e_agn, _ = btb.stitch_spectra(spectra, "agn", with_errors=True)
w_sf,  f_sf,  e_sf,  _ = btb.stitch_spectra(spectra, "sf",  with_errors=True)
print(f"median S/N per channel: AGN {np.nanmedian(f_agn / e_agn):.0f}, SF {np.nanmedian(f_sf / e_sf):.0f}")

ERR_DISPLAY_SCALE = 1       # exaggerate the shaded errors for display only (e.g. 10 if they are invisible)
fig = bplt.plot_component_spectra(w_agn, f_agn, w_sf, f_sf, smooth=2, err_agn=e_agn, err_sf=e_sf,
                                  err_scale=ERR_DISPLAY_SCALE)
proj.savefig(fig, "component_spectra")
proj.save({"agn": (w_agn, f_agn, e_agn), "sf": (w_sf, f_sf, e_sf), "per_band": spectra}, "step5_spectra")
proj.export_spectra({"AGN": (w_agn, f_agn, e_agn), "SF": (w_sf, f_sf, e_sf)})   # wavelength, flux, error

In [ ]:
proj.summary()

## Checklist before trusting the result

* AGN crosses on the nucleus in every band (Section 1).
* Spaxel fits sensible for every line; weak / blended lines removed (Sections 2-3).
* Geometry: fixed to reliable values, or well constrained by the fits; models not mirrored (Section 4).
* Line groups chosen from the maps, not only from the IP guess (Section 4).
* Dispersion histograms show two components for every decomposed line (Section 3).
* Residual maps without coherent structure (Section 5).
* Stacked map stable when lines are left out or weights changed (Section 6).

The method assumes two kinematic components with $\sigma_{\rm outflow} > \sigma_{\rm disc}$, a rotating disc and a
(hollow) biconical outflow. For other geometries, replace a component with an imported map (Section 4) or extend
`breaking_the_blend.models`. Each step is a plain function, so individual lines can be re-done by hand and swapped into
`line_maps` / `decomps` before stacking (see the end of `01_NGC5728.ipynb`).

## 8. Aperture spectra: nuclear spectrum vs. the AGN-dominated spectrum

A **nuclear spectrum** is extracted in a circular aperture of 0.3" radius centred on the AGN, in every band, and
compared with the AGN-dominated spectrum from the decomposition (Step 5). More apertures can be added to
`APERTURES`, each with its own radius and position, given as an (east, north) offset from the AGN in arcsec, so the
aperture covers the same sky region in every band. The extraction uses exact fractional pixel weights (like photutils'
`method="exact"`), the same error method, detector-gap and field-of-view handling as Step 5, and the bands are stitched
with **the same band scale factors** as the AGN-dominated spectrum, so the two can be compared and subtracted directly.
A band is skipped if less than 90% of the aperture lies inside its observed field.

The aperture spectra are **not corrected for the point-spread function**: a fixed 0.3" aperture loses a growing fraction
of a point source's flux towards longer wavelengths (MIRI/MRS FWHM ~0.3" at 5 um, ~1" at 28 um). The band-to-band
rescaling hides the jumps between bands but not the slope this introduces within each band.

In [ ]:
APERTURES = {   # name: radius [arcsec] and (east, north) offset of the centre from the AGN [arcsec]
    "nuclear": dict(radius_arcsec=0.3, offset_arcsec=(0.0, 0.0)),
    # "my region": dict(radius_arcsec=0.5, offset_arcsec=(1.0, -0.5)),
}
APERTURE_METHOD     = "exact"           # "exact" (fractional pixel weights) or "center" (whole pixels)
APERTURE_ERR_METHOD = SPECTRA_OPTIONS.get("err_method", "cube")   # same as Step 5

ap_spectra = btb.extract_apertures(cubes, APERTURES, method=APERTURE_METHOD, err_method=APERTURE_ERR_METHOD)
# stitch with exactly the band scale factors of the AGN-dominated spectrum (Step 5), so the two are comparable
_, _, _, AGN_SCALES = btb.stitch_spectra(spectra, "agn", with_errors=True, verbose=False)
ap_stitched = {}
for name in APERTURES:
    print(f"--- {name}")
    w, f, e, _ = btb.stitch_spectra(ap_spectra[name], "flux", with_errors=True, scales=AGN_SCALES)
    ap_stitched[name] = (w, f, e)
proj.save({"apertures": APERTURES, "per_band": ap_spectra, "stitched": ap_stitched}, "step6_aperture_spectra")
proj.export_spectra(ap_stitched, key="aperture_spectrum")          # wavelength, flux, error

The apertures on the AGN outflow fraction map:

In [ ]:
PC_PER_ARCSEC = next(iter(cubes.values())).pc_per_arcsec
fig = bplt.plot_apertures_on_map(stack, APERTURES, PC_PER_ARCSEC, which="f_out", lim_pc=MAP_LIM_PC,
                                 title=f"{GALAXY}: apertures")
proj.savefig(fig, "apertures_on_outflow_map")

The aperture spectra next to the AGN-dominated spectrum (errors shaded):

In [ ]:
to_plot = {f"AGN-dominated (f_out > {AGN_MIN:g})": (w_agn, f_agn, e_agn)}
to_plot.update({f'{n} ({APERTURES[n]["radius_arcsec"]:g}" aperture)': ap_stitched[n] for n in APERTURES})
fig = bplt.plot_spectra(to_plot)
proj.savefig(fig, "aperture_vs_agn_dominated_spectrum")

### AGN-dominated spectrum minus the nuclear spectrum

The AGN-dominated region contains both the AGN point source and the outflow. Removing the nuclear aperture spectrum (taken to
represent the AGN point source) from it leaves the emission of the rest of the AGN-dominated region - mainly the outflow.
This is done **per band on the pixels themselves** (the aperture's pixel weights are removed from the region's), then
stitched with the AGN-dominated scale factors. The result equals *AGN-dominated - nuclear*, with errors that do not
double-count the pixels the two share; only the part of the aperture inside the region is removed (printed per band).

No matching scale is applied. `NUCLEAR_SCALE` (default 1) multiplies the nuclear contribution before removing it. A 0.3\"
aperture misses part of a point source's flux - increasingly towards long wavelengths (PSF FWHM ~0.3\" at 5 um, ~1\" at
28 um) - so some AGN light remains in the result. A PSF aperture correction (> 1) would remove it; a single number is only
an approximation, as the correction depends on wavelength.

In [ ]:
SUB_APERTURE  = "nuclear"
NUCLEAR_SCALE = 1.0      # factor applied to the aperture before removing it (1 = as extracted; > 1 e.g. a PSF correction)

res_bands = btb.subtract_aperture(cubes, spectra, ap_spectra[SUB_APERTURE], "agn", scale=NUCLEAR_SCALE)
w_res, f_res, e_res, _ = btb.stitch_spectra(res_bands, "flux", with_errors=True, scales=AGN_SCALES)
print(f"residual < 0 in {100 * np.nanmean(f_res < 0):.0f}% of channels")

w_n, f_n, e_n = ap_stitched[SUB_APERTURE]
lab_ap = f'{SUB_APERTURE} ({APERTURES[SUB_APERTURE]["radius_arcsec"]:g}" aperture)' + (f" x {NUCLEAR_SCALE:g}" if NUCLEAR_SCALE != 1 else "")
fig, axes = plt.subplots(2, 1, figsize=(11, 8.5), sharex=True, constrained_layout=True)
bplt.plot_spectra({f"AGN-dominated (f_out > {AGN_MIN:g})": (w_agn, f_agn, e_agn),
                   lab_ap: (w_n, NUCLEAR_SCALE * f_n, NUCLEAR_SCALE * e_n)}, colors={lab_ap: "k"}, ax=axes[0])
bplt.plot_residual_spectrum(w_res, f_res, e_res, label=f"AGN-dominated - {SUB_APERTURE}", ax=axes[1])
proj.savefig(fig, "agn_dominated_minus_aperture")
proj.export_spectra({f"AGN_minus_{SUB_APERTURE}": (w_res, f_res, e_res)}, key="aperture_spectrum")